# Colab Full Reproduction

Notebook này ưu tiên Colab web. Thứ tự chạy được khóa là `Base -> Word2Vec -> CodeBERT`, sau đó mới đến `Enhanced`.

## Phase 0: Mount Drive

Cell này là bắt buộc. Toàn bộ output của các runner sẽ được lưu vào Drive dưới `VTI-Project2-runs`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Phase 1: Clone Latest `main`

Cell này là bắt buộc. Notebook luôn clone đầy đủ repo mới nhất từ `main` vào `/content/VTI-Project2`.

In [ ]:
REPO_URL = "https://github.com/tudzct/VTI-Project2.git"
REPO_DIR = "/content/VTI-Project2"

%cd /content
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd /content/VTI-Project2

In [ ]:
!pwd
!ls -lah /content
!ls -lah {REPO_DIR}
!git -C {REPO_DIR} branch --show-current
!git -C {REPO_DIR} rev-parse HEAD

## Phase 2: Install Dependencies

Các cell dưới đây dùng `%pip` để bám đúng kernel notebook. Chạy bootstrap trước, sau đó chỉ chạy thêm nhóm package tương ứng với mô hình bạn muốn chạy.

### Bootstrap

Bắt buộc. Nâng cấp `pip` cho kernel hiện tại.

In [ ]:
%pip install -q --upgrade pip

### Base Prerequisites

Bắt buộc cho notebook này. Nhóm package này phục vụ `Base` và là nền cho các bước kiểm tra dữ liệu.

In [ ]:
%pip install -q pandas numpy scipy scikit-learn

### Word2Vec Prerequisites

Chỉ cần khi chạy `Word2Vec` hoặc muốn chuẩn bị sẵn toàn bộ môi trường cho notebook.

In [ ]:
%pip install -q gensim tensorflow sentencepiece accelerate

### CodeBERT Prerequisites

Chỉ cần khi chạy `CodeBERT` hoặc muốn chuẩn bị sẵn toàn bộ môi trường cho notebook.

In [ ]:
%pip install -q torch transformers

In [ ]:
from pathlib import Path

BASE_OUT = '/content/drive/MyDrive/VTI-Project2-runs'
Path(BASE_OUT).mkdir(parents=True, exist_ok=True)
print(BASE_OUT)

## Phase 3: Verify Repo And Data

Cell này là bắt buộc. Notebook sẽ fail sớm nếu thiếu bộ dữ liệu full cần cho đường chạy chính.

In [ ]:
from pathlib import Path

repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    raise FileNotFoundError(f"Missing cloned repo: {repo_dir}")

required_full = [
    repo_dir / 'artifacts' / 'data_full' / 'train.csv.gz',
    repo_dir / 'artifacts' / 'data_full' / 'val.csv.gz',
    repo_dir / 'artifacts' / 'data_full' / 'test.csv.gz',
]
missing_full = [str(path) for path in required_full if not path.exists()]
if missing_full:
    raise FileNotFoundError('Missing required full dataset files:\n' + '\n'.join(missing_full))

print('Full dataset files:')
for path in required_full:
    print(f'present  {path}')

sample_files = [
    repo_dir / 'artifacts' / 'data_sample' / 'train.csv.gz',
    repo_dir / 'artifacts' / 'data_sample' / 'val.csv.gz',
    repo_dir / 'artifacts' / 'data_sample' / 'test.csv.gz',
]
print('\nSample dataset files:')
for path in sample_files:
    status = 'present' if path.exists() else 'missing'
    print(f'{status:7} {path}')

In [ ]:
!ls -lah /content/VTI-Project2/artifacts/data_full
!ls -lah /content/VTI-Project2/artifacts/data_sample

## Base

Main run cell: cell ngay dưới.
Fallback OOM cell: không cần ở pha này.
Output dir: `/content/drive/MyDrive/VTI-Project2-runs/base_full_p05`.

In [ ]:
!python /content/VTI-Project2/scripts/run_base_experiment.py \
  --train /content/VTI-Project2/artifacts/data_full/train.csv.gz \
  --val /content/VTI-Project2/artifacts/data_full/val.csv.gz \
  --test /content/VTI-Project2/artifacts/data_full/test.csv.gz \
  --output-dir {BASE_OUT}/base_full_p05 \
  --p-value-grid 0.5

In [ ]:
!ls -lah {BASE_OUT}/base_full_p05
!cat {BASE_OUT}/base_full_p05/metrics.json

## Word2Vec

Main run cell: cell ngay dưới.
Fallback OOM cell: cell comment ngay sau phần inspect.
Output dir: `/content/drive/MyDrive/VTI-Project2-runs/word2vec_full`.

In [ ]:
!python /content/VTI-Project2/scripts/run_word2vec_experiment.py \
  --train /content/VTI-Project2/artifacts/data_full/train.csv.gz \
  --val /content/VTI-Project2/artifacts/data_full/val.csv.gz \
  --test /content/VTI-Project2/artifacts/data_full/test.csv.gz \
  --output-dir {BASE_OUT}/word2vec_full \
  --max-train-rows 0 \
  --max-val-rows 0 \
  --max-test-rows 0 \
  --vector-size 300 \
  --w2v-epochs 30 \
  --nn-epochs 10 \
  --max-len 512 \
  --batch-size 128

In [ ]:
!ls -lah {BASE_OUT}/word2vec_full
!cat {BASE_OUT}/word2vec_full/metrics.json

Nếu Word2Vec bị OOM, dùng cell dự phòng bên dưới.

In [ ]:
# !python /content/VTI-Project2/scripts/run_word2vec_experiment.py \
#   --train /content/VTI-Project2/artifacts/data_full/train.csv.gz \
#   --val /content/VTI-Project2/artifacts/data_full/val.csv.gz \
#   --test /content/VTI-Project2/artifacts/data_full/test.csv.gz \
#   --output-dir {BASE_OUT}/word2vec_full \
#   --max-train-rows 0 \
#   --max-val-rows 0 \
#   --max-test-rows 0 \
#   --vector-size 200 \
#   --w2v-epochs 20 \
#   --nn-epochs 6 \
#   --max-len 256 \
#   --batch-size 64

## CodeBERT

Main run cell: `nvidia-smi` rồi cell run ngay dưới.
Fallback OOM cell: cell comment sau phần inspect.
Output dir: `/content/drive/MyDrive/VTI-Project2-runs/codebert_full`.

In [ ]:
!nvidia-smi

In [ ]:
!python /content/VTI-Project2/scripts/run_codebert_experiment.py \
  --train /content/VTI-Project2/artifacts/data_full/train.csv.gz \
  --val /content/VTI-Project2/artifacts/data_full/val.csv.gz \
  --test /content/VTI-Project2/artifacts/data_full/test.csv.gz \
  --output-dir {BASE_OUT}/codebert_full \
  --max-train-rows 0 \
  --max-val-rows 0 \
  --max-test-rows 0 \
  --epochs 2 \
  --batch-size 8 \
  --max-length 256 \
  --learning-rate 2e-5

In [ ]:
!ls -lah {BASE_OUT}/codebert_full
!cat {BASE_OUT}/codebert_full/metrics.json

Nếu CodeBERT bị OOM, dùng cell dự phòng bên dưới.

In [ ]:
# !python /content/VTI-Project2/scripts/run_codebert_experiment.py \
#   --train /content/VTI-Project2/artifacts/data_full/train.csv.gz \
#   --val /content/VTI-Project2/artifacts/data_full/val.csv.gz \
#   --test /content/VTI-Project2/artifacts/data_full/test.csv.gz \
#   --output-dir {BASE_OUT}/codebert_full \
#   --max-train-rows 0 \
#   --max-val-rows 0 \
#   --max-test-rows 0 \
#   --epochs 1 \
#   --batch-size 4 \
#   --max-length 256 \
#   --learning-rate 2e-5

## Enhanced (chạy sau)

Chỉ bắt đầu khi 3 mô hình nền đã có `raw_preds.csv` và `labels.csv`. Phần này vẫn là pha sau vì còn phụ thuộc CPG CSV upload từ local lên Drive.

In [ ]:
# !python /content/VTI-Project2/scripts/run_enhanced_notebook_compat.py \
#   --train "/content/drive/MyDrive/VTI-Project2-joern/train_subset_16384_with_cpg_fullrows.csv" \
#   --test "/content/drive/MyDrive/VTI-Project2-joern/test_full_with_cpg_fullrows.csv" \
#   --raw-preds {BASE_OUT}/base_full_p05/raw_preds.csv \
#   --labels {BASE_OUT}/base_full_p05/labels.csv \
#   --output-dir {BASE_OUT}/enhanced_base_full

In [ ]:
# !python /content/VTI-Project2/scripts/run_enhanced_notebook_compat.py \
#   --train "/content/drive/MyDrive/VTI-Project2-joern/train_subset_16384_with_cpg_fullrows.csv" \
#   --test "/content/drive/MyDrive/VTI-Project2-joern/test_full_with_cpg_fullrows.csv" \
#   --raw-preds {BASE_OUT}/word2vec_full/raw_preds.csv \
#   --labels {BASE_OUT}/word2vec_full/labels.csv \
#   --output-dir {BASE_OUT}/enhanced_word2vec_full

In [ ]:
# !python /content/VTI-Project2/scripts/run_enhanced_notebook_compat.py \
#   --train "/content/drive/MyDrive/VTI-Project2-joern/train_subset_16384_with_cpg_fullrows.csv" \
#   --test "/content/drive/MyDrive/VTI-Project2-joern/test_full_with_cpg_fullrows.csv" \
#   --raw-preds {BASE_OUT}/codebert_full/raw_preds.csv \
#   --labels {BASE_OUT}/codebert_full/labels.csv \
#   --output-dir {BASE_OUT}/enhanced_codebert_full

In [ ]:
!find {BASE_OUT} -maxdepth 2 -type f | sort